Object detection evaluation tools by Marek Bundzel, TU Kosice
marek.bundzel@tuke.sk
Primary use was for evaluation of object detection in archaeological Lidar data
Input files are geotifs with Ground truth and Prediction masks

## Bug fixes applied

**Fix 1 – `calculateClassifErrors`:** original code used `*2.0` producing a `float64` Errors array. Fixed to use `int16` arithmetic throughout.

**Fix 2 – `extendToRadius`:** original kernel zeroed the four corners of a 3×3 matrix, giving a diamond (L1) structuring element instead of a square (Chebyshev) one. At radius=10 this underestimates the dilation area by ~41 %, deflating the MOR10R metric. Fixed: full 3×3 all-ones kernel.

**Fix 3 – `analyzeAccuracyPerObject` label index:** original loop iterated `range(len(sizesGT))` (indices 0…N-1) and compared against cv2 label IDs (1…N), causing an off-by-one mismatch in `hited_sizes` / `no_hited_sizes`. Fixed: loop over actual label IDs `range(1, num_labels)` via a hit dict.

**Fix 4 – `analyzeAccuracyPerObject` uint8 cast:** `cv2.connectedComponentsWithStats` requires `uint8` input; rasterio often returns `int8`. Added explicit cast.

In [23]:
import numpy as np
import copy
import cv2
import matplotlib.pyplot as plt
import os
#import tensorflow as tf
#from keras.models import Model, load_model
import pickle
import glob
import numpy.ma as ma # For masked arrays
import imageio
import rasterio

In [24]:
#load ground truth from geotiff
#Ground truth (expected 0 = no object, 1 = object)
groundTruthPath = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/Maya_Labeling.tif"

with rasterio.open(groundTruthPath) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Shape:", src.shape)
    print("Bands:", src.count)
    print("Dtype:", src.dtypes)
    #print("Max:", src.max())
    GT = src.read(1)
    print("GT min:", GT.min())
    print("GT max:", GT.max())




CRS: EPSG:32616
Bounds: BoundingBox(left=215923.5, bottom=1916040.5, right=225714.5, top=1933132.5)
Shape: (17092, 9791)
Bands: 1
Dtype: ('float32',)
GT min: 0.0
GT max: 1.0


In [25]:
##load prediction from geotiff
#Ground truth (expected 0 = no object, 1 = object)
predictionPath = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/SPS_Labeled_Predictions.tif"

with rasterio.open(predictionPath) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Shape:", src.shape)
    print("Bands:", src.count)
    print("Dtype:", src.dtypes)
    #print("Max:", src.max())
    Prediction = src.read(1)
    print("Prediction min:", Prediction.min())
    print("Prediction max:", Prediction.max())


CRS: EPSG:32616
Bounds: BoundingBox(left=215923.5, bottom=1916040.5, right=225714.5, top=1933132.5)
Shape: (17092, 9791)
Bands: 1
Dtype: ('uint8',)
Prediction min: 0
Prediction max: 1


#Methods for calculating errors

calculateClassifErrors takes two matrices as input, assumed to be int, containing only 1 and 0. The matrices are the prediction and ground truth: GT. The output is a matrix Errors, which has 0 for true negative, 2 for true positive, -1 for false negative and 1 for false positive.

In [26]:
def calculateClassifErrors(Prediction, GT):
    """
    Returns an int16 array:
      0  = True Negative  (TN)
      2  = True Positive  (TP)
     -1  = False Negative (FN)
      1  = False Positive (FP)

    Bug fix: original code used *2.0 which produced a float64 intermediate,
    causing the Errors array to be float64 instead of integer.
    Now uses integer arithmetic throughout.
    """
    P = Prediction.astype(np.int16)
    G = GT.astype(np.int16)
    Errors = ((P + G) > 1).astype(np.int16) * 2
    Errors = Errors + (P - G)
    return Errors.astype(np.int16)


In [27]:
Errors = calculateClassifErrors(Prediction,GT)
print("Errors shape:", Errors.shape)
print("Errors min:", Errors.min())
print("Errors max:", Errors.max())

Errors shape: (17092, 9791)
Errors min: -1
Errors max: 2


Visualization of Errors

In [28]:
def saveErrorsAsGeotiff(Errors, reference_tif_path, output_path):

    # Ensure correct dtype
    Errors = Errors.astype(np.int8)

    # --- Load reference georeferencing ---
    with rasterio.open(reference_tif_path) as ref:
        profile = ref.profile.copy()

    # --- Update profile for single-band int8 raster ---
    profile.update({
        "driver": "GTiff",
        "count": 1,
        "dtype": "int8",
        "compress": "deflate",
        "predictor": 2,
        "zlevel": 9,
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
        "nodata": None
    })

    # --- Write raster ---
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(Errors, 1)

        # --- Store class labels as metadata (for reference) ---
        dst.update_tags(1,
            **{
                "-1": "False Negative",
                "0": "True Negative",
                "1": "False Positive",
                "2": "True Positive"
            }
        )

    print("Saved:", output_path)

def showErrors(Errors):
    from matplotlib.figure import Figure
    from matplotlib.colors import LinearSegmentedColormap
    colors = [(0, 0, 1), (0, 0, 0), (1, 0, 0), (1, 1, 1)]  # Blue, Black, Red, White
    cmap_name = 'toShow'
    cmapToShow = LinearSegmentedColormap.from_list(cmap_name, colors, N=4)
      
    fig = Figure(figsize=(30, 20))
    plt.imshow(Errors, cmap=cmapToShow)
    plt.savefig('classifVsGT.png', dpi=1200)
    plt.show()

    print('black: 0(TN), white: 2(TP), blue: -1(FN), red: 1(FP)')

def saveErrorsAsPNG(outfilePath, Errors):
    colors = [(0, 0, 1), (0, 0, 0), (1, 0, 0), (1, 1, 1)]  # Blue, Black, Red, White
    img = np.zeros((Errors.shape[0],Errors.shape[1],3),dtype = np.dtype('uint8'))
    img[:,:,0] = (Errors == 2) + (Errors == 1)
    img[:,:,1] = (Errors == 2)
    img[:,:,2] = (Errors == 2) + (Errors == -1)
    img = img*255

    imageio.imwrite(outfilePath, img)

def make_error_path(tif_path):
    #makes a path to a geotiff that has the same location as the tif_path but has _Errors appended
    # Split into directory and filename
    directory, filename = os.path.split(tif_path)

    # Split filename into name and extension
    name, ext = os.path.splitext(filename)

    # Create new filename
    new_filename = f"Errors_{name}{ext}"

    # Join back to full path
    return os.path.join(directory, new_filename)


In [29]:
#path to the output geotiff
outputPath = make_error_path(predictionPath)
# Save the errors as geotiff, symbology =
# "-1": "False Negative",
# "0": "True Negative",
# "1": "False Positive",
# "2": "True Positive"
#saveErrorsAsGeotiff(Errors, reference_tif_path=groundTruthPath, output_path = outputPath)


The following method is used to calculate statistical indicators from the Confusion matrix:
For details, see: https://en.wikipedia.org/wiki/Confusion_matrix


In [30]:
def calculateStatistics(Errors):
    # ('black: 0(TN), white: 2(TP), blue: -1(FN), red: 1(FP)')
    # reconstruct predicted and ground truth
    predicted = np.zeros(Errors.shape, dtype='uint8')
    GT = np.zeros(Errors.shape, dtype='uint8')
    predicted = predicted + (Errors==2) + (Errors==1)
    GT = GT + (Errors==2) + (Errors==-1)

    TN = np.sum(Errors==0)
    TP = np.sum(Errors==2)
    FN = np.sum(Errors==-1)
    FP = np.sum(Errors==1)

    P = TP + FN  # Positive examples
    N = TN + FP  # Negative examples

    TPR = TP/(TP+FN)  # Sensitivity, true positive rate
    TNR = TN/(TN+FP)  # Specificity, true negative rate

    BalancedAccuracy = (TPR + TNR)/2
    Accuracy = (TP + TN)/(P + N)

    MOR5R = countMisclassifiedOutOfRadius(Errors, 5)/(P+N)
    MOR10R = countMisclassifiedOutOfRadius(Errors, 10)/(P+N)
    MOR15R = countMisclassifiedOutOfRadius(Errors, 15)/(P+N)
    MOR20R = countMisclassifiedOutOfRadius(Errors, 20)/(P+N)
    MOR25R = countMisclassifiedOutOfRadius(Errors, 25)/(P+N)
    MOR50R = countMisclassifiedOutOfRadius(Errors, 50)/(P+N)

    IoU_fore, IoU_bgr, IoU_ave  = intersectionOverUnion(predicted, GT)

    PPV = TP/(TP + FP)  # Positive predictive value, precision
    NPV = TN/(TN + FN)  # Negative predictive value

    F1 = 2*TP/(2*TP + FP +FN)

    TN = TN.astype('float64')
    TP = TP.astype('float64')
    FN = FN.astype('float64')
    FP = FP.astype('float64')
    MCC = (TP * TN - FP * FN)/np.sqrt(((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN)))  # Matthews correlation coef.

    Statistics = dict(
        BalancedAccuracy = BalancedAccuracy,
        Accuracy = Accuracy,
        TN = TN,
        TP = TP,
        FN = FN,
        FP = FP,
        TPR = TPR,
        TNR = TNR,
        PPV = PPV,
        NPV = NPV,
        F1 = F1,
        MOR5R = MOR5R,
        MOR10R = MOR10R,
        MOR15R = MOR15R,
        MOR20R = MOR20R,
        MOR25R = MOR25R,
        MOR50R = MOR50R,
        IoU_fore = IoU_fore,
        IoU_bgr = IoU_bgr,
        IoU_ave = IoU_ave,
        MCC = MCC
    )

    return Statistics

def countMisclassifiedOutOfRadius(errors, radius):
    TP = errors == 2
    Misclassified = np.logical_or((errors == -1),(errors == 1))
    TP = extendToRadius(TP,radius)

    return np.sum(Misclassified) - np.sum(np.logical_and(Misclassified,TP))

def intersectionOverUnion(prediction, groundTruth):
    pr = prediction > 0
    gt = groundTruth > 0
    intersection = np.logical_and(pr, gt)
    union = np.logical_or(pr, gt)
    iou_fore = np.sum(intersection) / np.sum(union)
    pr = prediction < 1
    gt = groundTruth < 1
    intersection = np.logical_and(pr, gt)
    union = np.logical_or(pr, gt)
    iou_bgr = np.sum(intersection) / np.sum(union)
    iou_ave = (iou_fore + iou_bgr)/2

    return iou_fore, iou_bgr, iou_ave

def extendToRadius(binary_img, radius):
    """
    Dilate the binary image by the given radius.

    Bug fix: the original code used a diamond-shaped (cross) structuring element
    by zeroing the four corners of a 3x3 kernel. This produced a rotated-square
    (L1/Manhattan) neighbourhood rather than the intended square (Chebyshev)
    neighbourhood. At radius=10 a diamond covers ~41% fewer pixels than a square,
    systematically underestimating the MOR10R metric.
    Fixed: use a full 3x3 all-ones kernel for a square (Chebyshev) dilation.
    """
    img = binary_img.astype('uint8')
    # Full 3x3 all-ones kernel -> square structuring element
    kernel = np.ones((3, 3), np.uint8)

    extended = cv2.dilate(img, kernel, iterations=radius)

    return extended == 1


In [31]:
Statistics = calculateStatistics(Errors)
print(Statistics)

{'BalancedAccuracy': np.float64(0.6802138490437045), 'Accuracy': np.float64(0.9970209403206157), 'TN': np.float64(166685793.0), 'TP': np.float64(163440.0), 'FN': np.float64(288443.0), 'FP': np.float64(210096.0), 'TPR': np.float64(0.3616865427555363), 'TNR': np.float64(0.9987411553318728), 'PPV': np.float64(0.43754818812644564), 'NPV': np.float64(0.9982725299009603), 'F1': np.float64(0.396017053157245), 'MOR5R': np.float64(0.002238709219265853), 'MOR10R': np.float64(0.002087019120876016), 'MOR15R': np.float64(0.0019566857454188275), 'MOR20R': np.float64(0.0018488922577349878), 'MOR25R': np.float64(0.001765909378225842), 'MOR50R': np.float64(0.0015194704832998912), 'IoU_fore': np.float64(0.24689604957256953), 'IoU_bgr': np.float64(0.9970180279812345), 'IoU_ave': np.float64(0.621957038776902), 'MCC': np.float64(0.3963355373637591)}


Evaluation of Hits and Misses plus histograms
Object is considered hit if there is at least one predicted pixel within the patch of the label
False positive object is when there is no pixel of the label within the patch of the prediction


In [32]:
def analyzeAccuracyPerObject(GT, Prediction, connectivity=8, NoOfBins=400):
    # Bug fix: cv2.connectedComponentsWithStats requires uint8 input.
    # If GT/Prediction arrive as int8 (common from rasterio) the function
    # raises a cryptic OpenCV error. Cast explicitly.
    GT_u8   = GT.astype(np.uint8)
    Pred_u8 = Prediction.astype(np.uint8)

    num_labels, labeledGT, stats, centroids = cv2.connectedComponentsWithStats(
        GT_u8, cv2.CC_STAT_AREA, connectivity=connectivity)
    # stats — 2D array of shape (num_labels, 5).
    # Each row corresponds to one label and contains:
    # cv2.CC_STAT_LEFT   x of bounding box
    # cv2.CC_STAT_TOP    y of bounding box
    # cv2.CC_STAT_WIDTH  width of bounding box
    # cv2.CC_STAT_HEIGHT height of bounding box
    # cv2.CC_STAT_AREA   area in pixels
    no_of_GT_objects = num_labels - 1
    print("Number of GT objects")
    print(no_of_GT_objects)
    sizesGT = stats[1:, 4]  # skip background label 0
    ave_size_GT_object = np.sum(sizesGT)/len(sizesGT)
    min_size_GT_object = np.min(sizesGT)
    max_size_GT_object = np.max(sizesGT)
    print("Average size of GT object")
    print(ave_size_GT_object)
    print("Minimal size of GT object")
    print(min_size_GT_object)
    print("Maximal size of GT object")
    print(max_size_GT_object)

    # Evaluating which objects were hit
    hit_img = labeledGT * Pred_u8
    hits_unique = np.unique(hit_img, return_counts=True)
    hits = hits_unique[0][1:]       # label IDs that were hit (excluding background 0)
    hit_counts = hits_unique[1][1:]
    no_of_GT_objects_detected = len(hits)
    print("Number of GT objects that were detected")
    print(no_of_GT_objects_detected)
    print("Average number of hit pixels per GT object")
    ave_hit_pixels_per_object = np.sum(hit_counts)/len(hit_counts)
    print(ave_hit_pixels_per_object)

    hited_sizes    = []
    no_hited_sizes = []

    # Bug fix: the original loop used range(len(sizesGT)) with i starting at 0,
    # but cv2 labels start at 1 (0 is background). This caused an off-by-one
    # mismatch: the first object (label 1) was never found in `hits` (which
    # contains values >= 1), and the last object was never checked.
    # Fix: build a dict of hit label IDs, then iterate over actual label IDs 1..N.
    hit_dict = {int(hits[i]): int(hit_counts[i]) for i in range(len(hits))}

    for label_id in range(1, num_labels):      # label 0 is background
        size = int(stats[label_id, 4])
        if label_id in hit_dict:
            hited_sizes.append(size)
        else:
            no_hited_sizes.append(size)

    sizes = np.sort(sizesGT)
    bns = np.linspace(np.min(sizes), np.max(sizes)+1, num=NoOfBins)
    plt.figure(figsize=(16,8))
    plt.hist(sizes, bins=bns, label='All Objects')
    plt.hist(hited_sizes, bins=bns, label='Detected Objects')
    plt.legend(prop={'size': 10})
    plt.ylabel('No. of Objects & No. of Objects Detected')
    plt.xlabel('Sizes')
    plt.title('Histogram of Total and Detected Objects Counts')

    num_labelsPrediction, labeledPrediction, stats, centroids = cv2.connectedComponentsWithStats(
        Pred_u8, cv2.CC_STAT_AREA, connectivity=connectivity)
    no_of_Prediction_objects = num_labelsPrediction - 1
    print("Number of Predicted objects")
    print(no_of_Prediction_objects)
    sizesPrediction = stats[1:, 4]
    ave_size_Prediction_object = np.sum(sizesPrediction)/len(sizesPrediction)
    min_size_Prediction_object = np.min(sizesPrediction)
    max_size_Prediction_object = np.max(sizesPrediction)
    print("Average size of Predicted object")
    print(ave_size_Prediction_object)
    print("Minimal size of Predicted object")
    print(min_size_Prediction_object)
    print("Maximal size of Predicted object")
    print(max_size_Prediction_object)

    # Evaluating false positive detections
    hit_img = labeledPrediction * GT_u8
    hits_unique = np.unique(hit_img, return_counts=True)
    hits = hits_unique[0][1:]
    hit_counts = hits_unique[1][1:]
    no_of_Prediction_objects_TP = len(hits)
    print("Number of True Positive Prediction objects")
    print(no_of_Prediction_objects_TP)
    no_of_Prediction_objects_FP = no_of_Prediction_objects - no_of_Prediction_objects_TP
    print("Number of False Positive Prediction objects")
    print(no_of_Prediction_objects_FP)


In [33]:
#analyzeAccuracyPerObject(GT,Prediction, connectivity=8, NoOfBins=200)